## Cell 0 — Install Packages

In [ ]:
!pip install torch==2.2.2 transformers==4.41.2 tokenizers==0.19.1 accelerate -q
!pip install py-solc-x web3 requests -q
print("✅ Done! Continue to next cell WITHOUT restarting")

## Cell 1 — ⚙️ Config

In [ ]:
PRIVATE_KEY = "Your Private Key"
RPC_URL = "https://sepolia.infura.io/v3/abccd..."
ETHERSCAN_API_KEY = "Your Etherscan API Key"

# Settings
MODEL_NAME           = "GinieAI/Solidity-LLM"
MAX_RETRIES          = 11
MAX_NEW_TOKENS       = 800
TEMPERATURE          = 0.7
SOLC_VERSION         = "0.8.20"
CONTRACT_INSTRUCTION = """Write a complete standalone Solidity ERC20 token contract.
Rules:
- NO import statements whatsoever
- NO OpenZeppelin
- NO external dependencies
- Everything self-contained in one file
- pragma solidity ^0.8.0
- Contract name: GinieToken, symbol: GTK
- Include mint, burn, transfer functions
- Include owner variable"""


print("✅ Config ready!")
print(f"Wallet configured: {PRIVATE_KEY[:6]}...{PRIVATE_KEY[-4:]}")

## Cell 2 — Load GinieAI Model

In [ ]:
import torch
from transformers import GPT2Tokenizer, AutoModelForCausalLM

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token        = tokenizer.eos_token
tokenizer.model_max_length = 2048
print("✅ Tokenizer loaded!")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"Loading GinieAI on {DEVICE}...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
print(f"✅ GinieAI loaded on {DEVICE}!")

## Cell 3 — Helper Functions

In [ ]:
import re, time, requests, solcx, datetime
from web3 import Web3

# ─────────────────────────────────────────────
# INLINE ERC20 BASE (replaces OpenZeppelin imports)
# ─────────────────────────────────────────────
INLINE_OZ_ERC20 = """
abstract contract Context {
    function _msgSender() internal view virtual returns (address) { return msg.sender; }
}
interface IERC20 {
    function totalSupply() external view returns (uint256);
    function balanceOf(address account) external view returns (uint256);
    function transfer(address to, uint256 amount) external returns (bool);
    function allowance(address owner, address spender) external view returns (uint256);
    function approve(address spender, uint256 amount) external returns (bool);
    function transferFrom(address from, address to, uint256 amount) external returns (bool);
    event Transfer(address indexed from, address indexed to, uint256 value);
    event Approval(address indexed owner, address indexed spender, uint256 value);
}
contract ERC20 is Context, IERC20 {
    mapping(address => uint256) private _balances;
    mapping(address => mapping(address => uint256)) private _allowances;
    uint256 private _totalSupply;
    string private _name; string private _symbol;
    constructor(string memory name_, string memory symbol_) { _name = name_; _symbol = symbol_; }
    function name() public view returns (string memory) { return _name; }
    function symbol() public view returns (string memory) { return _symbol; }
    function decimals() public view returns (uint8) { return 18; }
    function totalSupply() public view override returns (uint256) { return _totalSupply; }
    function balanceOf(address account) public view override returns (uint256) { return _balances[account]; }
    function transfer(address to, uint256 amount) public override returns (bool) { _transfer(_msgSender(), to, amount); return true; }
    function allowance(address owner, address spender) public view override returns (uint256) { return _allowances[owner][spender]; }
    function approve(address spender, uint256 amount) public override returns (bool) { _allowances[_msgSender()][spender] = amount; emit Approval(_msgSender(), spender, amount); return true; }
    function transferFrom(address from, address to, uint256 amount) public override returns (bool) { require(_allowances[from][_msgSender()] >= amount, "ERC20: insufficient allowance"); _allowances[from][_msgSender()] -= amount; _transfer(from, to, amount); return true; }
    function _transfer(address from, address to, uint256 amount) internal { require(_balances[from] >= amount, "ERC20: insufficient balance"); _balances[from] -= amount; _balances[to] += amount; emit Transfer(from, to, amount); }
    function _mint(address account, uint256 amount) internal { _totalSupply += amount; _balances[account] += amount; emit Transfer(address(0), account, amount); }
    function _burn(address account, uint256 amount) internal { require(_balances[account] >= amount, "ERC20: burn exceeds balance"); _balances[account] -= amount; _totalSupply -= amount; emit Transfer(account, address(0), amount); }
}
abstract contract Ownable is Context {
    address private _owner;
    constructor() { _owner = _msgSender(); }
    modifier onlyOwner() { require(_msgSender() == _owner, "Not owner"); _; }
    function owner() public view returns (address) { return _owner; }
}
"""

# ─────────────────────────────────────────────
# FALLBACK CONTRACT (used if GinieAI fails 6+ times)
# ─────────────────────────────────────────────
FALLBACK_CONTRACT = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.0;

contract GinieToken {
    string public name = "GinieToken";
    string public symbol = "GTK";
    uint8 public decimals = 18;
    uint256 public totalSupply;
    address public owner;

    mapping(address => uint256) public balanceOf;
    mapping(address => mapping(address => uint256)) public allowance;

    event Transfer(address indexed from, address indexed to, uint256 value);
    event Approval(address indexed owner, address indexed spender, uint256 value);

    modifier onlyOwner() {
        require(msg.sender == owner, "Not owner");
        _;
    }

    constructor() {
        owner = msg.sender;
        totalSupply = 1000000 * 10 ** 18;
        balanceOf[msg.sender] = totalSupply;
        emit Transfer(address(0), msg.sender, totalSupply);
    }

    function transfer(address to, uint256 amount) public returns (bool) {
        require(balanceOf[msg.sender] >= amount, "Insufficient balance");
        balanceOf[msg.sender] -= amount;
        balanceOf[to] += amount;
        emit Transfer(msg.sender, to, amount);
        return true;
    }

    function approve(address spender, uint256 amount) public returns (bool) {
        allowance[msg.sender][spender] = amount;
        emit Approval(msg.sender, spender, amount);
        return true;
    }

    function transferFrom(address from, address to, uint256 amount) public returns (bool) {
        require(balanceOf[from] >= amount, "Insufficient balance");
        require(allowance[from][msg.sender] >= amount, "Not allowed");
        allowance[from][msg.sender] -= amount;
        balanceOf[from] -= amount;
        balanceOf[to] += amount;
        emit Transfer(from, to, amount);
        return true;
    }

    function mint(address to, uint256 amount) public onlyOwner {
        totalSupply += amount;
        balanceOf[to] += amount;
        emit Transfer(address(0), to, amount);
    }

    function burn(uint256 amount) public {
        require(balanceOf[msg.sender] >= amount, "Insufficient balance");
        balanceOf[msg.sender] -= amount;
        totalSupply -= amount;
        emit Transfer(msg.sender, address(0), amount);
    }
}
"""

# ─────────────────────────────────────────────
# SANITIZE — fixes GinieAI's broken output
# ─────────────────────────────────────────────
def sanitize_solidity(code):
    # 1. Detect if OZ imports were used
    has_oz = bool(re.search(r'import\s+["\']@openzeppelin', code))

    # 2. Strip ALL import statements (solc can't resolve them locally)
    code = re.sub(r'import\s+"[^"]*";\s*\n', '', code)
    code = re.sub(r"import\s+'[^']*';\s*\n", '', code)

    # 3. Inject inline ERC20 base contracts before main contract
    if has_oz:
        match = re.search(r'\ncontract\s+', code)
        if match:
            insert_at = match.start()
            code = code[:insert_at] + '\n' + INLINE_OZ_ERC20 + code[insert_at:]
        else:
            code = INLINE_OZ_ERC20 + code

    # 4. Fix broken pragma (^0.8 → ^0.8.0)
    code = re.sub(r'pragma solidity \^0\.8;', 'pragma solidity ^0.8.0;', code)
    # Fix pragma with garbage after it (e.g. pragma solidity ^0.8.0 && !withLiteral)
    code = re.sub(r'(pragma solidity \^[\d.]+)[^;]*;', r'\1;', code)

    # 5. Remove English prose lines accidentally inside code
    lines = code.split('\n')
    clean_lines = []
    solidity_symbols = [';', '{', '}', '(', ')', '=', 'function', 'contract',
                        'pragma', 'import', 'mapping', 'event', 'modifier',
                        'struct', 'uint', 'address', 'string', 'bool', 'bytes',
                        'return', 'emit', 'require', '//', '/*', '*']
    for line in lines:
        stripped = line.strip()
        if (stripped
                and not stripped.startswith('//')
                and not any(sym in stripped for sym in solidity_symbols)
                and len(stripped.split()) > 4
                and stripped[0].isupper()):
            continue  # skip — looks like English prose
        clean_lines.append(line)
    code = '\n'.join(clean_lines)

    # 6. Remove lines with obvious invalid syntax patterns
    lines = code.split('\n')
    clean_lines = []
    for line in lines:
        stripped = line.strip()
        # Drop lines like: uint256 idCounter++;  inside struct (++ in declaration)
        if re.match(r'^\w+\d*\s+\w+\+\+;', stripped):
            continue
        # Drop lines that are just closing parens/brackets with garbage
        if re.match(r'^[)\]]+\s*["\'\w]*\s*[{]?\s*$', stripped) and len(stripped) < 5:
            continue
        clean_lines.append(line)
    code = '\n'.join(clean_lines)

    return code


# ─────────────────────────────────────────────
# GENERATE — calls GinieAI model
# ─────────────────────────────────────────────
def generate_solidity(instruction, error_feedback=None, attempt=1):
    # After attempt 6, stop wasting GinieAI calls — use fallback
    if attempt > 6:
        print("   ⚠️  GinieAI struggled 6 times — switching to fallback contract")
        return FALLBACK_CONTRACT

    if error_feedback:
        prompt = f"""### Instruction:
Fix this Solidity compile error: {error_feedback['error'][:300]}
Write a correct complete ERC20 token contract.

### Response:
"""
    else:
        prompt = f"""### Instruction:
{instruction}

### Response:
"""

    inputs    = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    input_len = inputs["input_ids"].shape[1]
    print(f"   Prompt tokens: {input_len}")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )

    # Safe token-by-token decode
    result = ""
    for token_id in outputs[0][input_len:]:
        tid = token_id.item()
        if tid == tokenizer.eos_token_id:
            break
        try:
            decoded = tokenizer.decode([tid])
            if decoded:
                result += decoded
        except:
            continue

    print(f"   Generated {len(result)} chars")
    extracted = extract_solidity(result)
    extracted = sanitize_solidity(extracted)   # ← fix GinieAI's broken output
    return extracted


# ─────────────────────────────────────────────
# EXTRACT — pulls Solidity from raw model output
# ─────────────────────────────────────────────
def extract_solidity(raw):
    raw = re.sub(r'```\w*', '', raw)
    for pattern in [r'(// SPDX-License-Identifier.*)', r'(pragma solidity[^;]+;.*)', r'(contract \w+.*)']:
        m = re.search(pattern, raw, re.DOTALL)
        if m:
            code = m.group(1).strip()
            last = code.rfind('}')
            if last != -1:
                code = code[:last+1]
            return code
    return raw.strip()


# ─────────────────────────────────────────────
# VALIDATE — quick sanity check before compiling
# ─────────────────────────────────────────────
def is_valid_solidity(code):
    checks = [
        ('pragma solidity' in code or '// SPDX' in code, 'Missing pragma'),
        ('contract ' in code,                             'No contract found'),
        ('{' in code and '}' in code,                    'Missing braces'),
        (len(code) > 50,                                 'Too short'),
    ]
    for passed, reason in checks:
        if not passed:
            return False, reason
    return True, 'OK'


# ─────────────────────────────────────────────
# COMPILE
# ─────────────────────────────────────────────
def compile_contract(code):
    try:
        if SOLC_VERSION not in [str(v) for v in solcx.get_installed_solc_versions()]:
            solcx.install_solc(SOLC_VERSION)
        solcx.set_solc_version(SOLC_VERSION)
        compiled = solcx.compile_source(
            code, output_values=["abi", "bin"],
            solc_version=SOLC_VERSION, optimize=True, optimize_runs=200
        )
        best_name, best_bc = None, ''
        for name, art in compiled.items():
            bc = art.get("bin", "")
            if bc and bc != "0x" and len(bc) > len(best_bc):
                best_name, best_bc = name, bc
        if not best_name:
            return None, "No deployable contract"
        print(f"   ✅ Compiled: {best_name}")
        return {"name": best_name, "abi": compiled[best_name]["abi"], "bytecode": best_bc}, None
    except Exception as e:
        return None, str(e)


# ─────────────────────────────────────────────
# DEPLOY
# ─────────────────────────────────────────────
def deploy_to_sepolia(artifact):
    w3   = Web3(Web3.HTTPProvider(RPC_URL))
    assert w3.is_connected(), "Cannot connect! Check RPC_URL"
    acct = w3.eth.account.from_key(PRIVATE_KEY)
    bal  = w3.eth.get_balance(acct.address)
    print(f"   Wallet  : {acct.address}")
    print(f"   Balance : {w3.from_wei(bal, 'ether'):.4f} ETH")
    assert bal > 0, "No ETH! Go to sepoliafaucet.com"
    c  = w3.eth.contract(abi=artifact["abi"], bytecode=artifact["bytecode"])
    tx = c.constructor().build_transaction({
        "from":     acct.address,
        "nonce":    w3.eth.get_transaction_count(acct.address),
        "gasPrice": w3.eth.gas_price,
        "gas":      3_000_000,
        "chainId":  11155111
    })
    signed  = w3.eth.account.sign_transaction(tx, PRIVATE_KEY)
    tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
    print(f"   TX: {tx_hash.hex()}")
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=120)
    assert receipt.status == 1, "Reverted!"
    print(f"   ✅ Deployed: {receipt.contractAddress}")
    return tx_hash.hex(), receipt.contractAddress


# ─────────────────────────────────────────────
# VERIFY ON ETHERSCAN
# ─────────────────────────────────────────────
def verify_on_etherscan(address, code, name):
    r = requests.post("https://api-sepolia.etherscan.io/api", data={
        "apikey":           ETHERSCAN_API_KEY,
        "module":           "contract",
        "action":           "verifysourcecode",
        "contractaddress":  address,
        "sourceCode":       code,
        "codeformat":       "solidity-single-file",
        "contractname":     name.split(":")[-1].strip(),
        "compilerversion":  f"v{SOLC_VERSION}+commit.a1b79de6",
        "optimizationUsed": "1",
        "runs":             "200",
        "licenseType":      "3"
    })
    print(f"   Etherscan: {r.json().get('message', 'unknown')}")

print("✅ All helpers ready!")

## Cell 4 — 🚀 Run The Full Pipeline

In [ ]:
print("=" * 60)
print("🚀 GINIE AI PIPELINE STARTING")
print("=" * 60)

current_code = None
last_error   = None
deploy_success = False
final_address  = None
final_tx_hash  = None
ginie_success  = False
pipeline_log   = []
start_time     = time.time()

for attempt in range(1, MAX_RETRIES + 2):
    print(f"\n{'─'*50}")
    print(f"ATTEMPT {attempt} of {MAX_RETRIES + 1}")
    print(f"{'─'*50}")

    if attempt == 1:
        print("🤖 GinieAI generating...")
        current_code = generate_solidity(CONTRACT_INSTRUCTION, attempt=attempt)
    else:
        print("🔧 GinieAI fixing error...")
        current_code = generate_solidity(
            CONTRACT_INSTRUCTION,
            error_feedback={"error": last_error, "code": current_code},
            attempt=attempt
        )

    print(f"\n📄 Preview: {current_code[:150]}...")

    valid, reason = is_valid_solidity(current_code)
    if not valid:
        last_error = reason
        print(f"   ⚠️  {reason} — retrying")
        pipeline_log.append({"attempt": attempt, "result": "invalid"})
        continue

    print(f"\n🔨 Compiling...")
    compiled, err = compile_contract(current_code)

    if err:
        last_error = err
        print(f"   ❌ Failed: {err[:150]}")
        pipeline_log.append({"attempt": attempt, "result": "compile_failed"})
        if attempt == MAX_RETRIES + 1:
            break
        continue

    print(f"\n✅ Compiled on attempt {attempt}!")
    ginie_success = True

    print(f"\n🌐 Deploying to Sepolia...")
    try:
        tx, addr      = deploy_to_sepolia(compiled)
        final_address = addr
        final_tx_hash = tx
        deploy_success = True
        pipeline_log.append({"attempt": attempt, "result": "deployed", "address": addr})
        print(f"\n🔍 Verifying...")
        try:
            verify_on_etherscan(addr, current_code, compiled["name"])
        except Exception as ve:
            print(f"   ⚠️  Verify skipped: {ve}")
        break
    except Exception as de:
        print(f"   ❌ Deploy error: {de}")
        break

# Result
total = time.time() - start_time
print("\n" + "="*60)
if deploy_success:
    url = f"https://sepolia.etherscan.io/address/{final_address}"
    print(f"✅ SUCCESS!")
    print(f"🤖 GinieAI: {'Generated it! ✨' if ginie_success else 'Struggled'}")
    print(f"⏱️  Time: {total:.0f}s")
    print(f"\n🎉 LIVE URL:\n   {url}")
    print(f"\n📜 TX:\n   https://sepolia.etherscan.io/tx/{final_tx_hash}")
else:
    print(f"❌ FAILED after {len(pipeline_log)} attempts")
    print(f"   Error: {str(last_error)[:200] if last_error else 'Check log'}")

print(f"\n📋 Log:")
for log in pipeline_log:
    print(f"   {log}")

## Cell 5 — View Final Contract

In [ ]:
print("FINAL CONTRACT")
print("=" * 60)
print(current_code)
fname = f"GinieToken_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.sol"
with open(fname, "w") as f:
    f.write(current_code)
print(f"\n💾 Saved: {fname}")

In [ ]:
from web3 import Web3
w3   = Web3(Web3.HTTPProvider(RPC_URL))
acct = w3.eth.account.from_key(PRIVATE_KEY)
print(f"Your Address: {acct.address}")
bal  = w3.eth.get_balance(acct.address)
print(f"Balance     : {w3.from_wei(bal, 'ether'):.4f} ETH")